[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLAlchemy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)

# SQL Expressions


## What you will be able to do

Build `select`, `insert`, `update` and `delete` statements from the college's tables, with conditions
combined the way SQL means them, functions, grouping and joins that follow the foreign keys. Get rows
back from a write with `returning()`, count what a write changed, write an upsert for an import that
may run twice, and run one statement with many sets of values. Read the SQL any statement becomes,
and recognize the conditions that Python quietly turns into something else.


## The idea

### The problem

The registrar's application has grown well past the search page. In January it admits new students
and enrolls them. Every evening it moves students between sections. At the end of every term it
records a few hundred grades, and a nightly job imports grades from the departments' own systems,
and sometimes runs twice. Written with `text()`, every one of these is a string, and a string cannot
be extended: a report that needs one more condition for one caller needs another string, and an
import that runs twice fails on every row it already wrote.

Python also makes one mistake easy. A condition written as `a and b`, with `a` and `b` two
comparisons such as `students.c.program == "Biology"`, reads like a condition with two parts, runs
without an error, and becomes SQL with one part. Python's `and` is not a function a library can
change, and it decides what to return before SQLAlchemy is asked. The mistake is invisible in the
code, visible in the SQL, and silent in the results, which simply hold more rows than they should.

### What a SQL expression is

> A **SQL expression** is a piece of SQL built from Python objects. A column compared with a value,
> `students.c.program == "History"`, is a condition that becomes `students.program = ?`, not a Python
> `True` or `False`. Conditions combine with **`&`**, **`|`** and **`~`**, or with **`and_()`**,
> **`or_()`** and **`not_()`**, and **`func`** calls any SQL function, as in `func.count()`.
> **`select()`**, **`insert()`**, **`update()`** and **`delete()`** build the four statements:
> `.where()` filters, `.values()` gives the values to write, `.order_by()`, `.group_by()`, `.limit()`
> and `.join()` shape a query, and **`.returning()`** asks a write for rows back. A **`bindparam()`**
> names a value that is supplied when the statement runs, so one statement can run with many sets of
> values.

### Why it works that way

- **Operators on columns build SQL.** `==`, `!=`, `<`, `>=` and `+` return expressions, and
  `in_()`, `like()`, `startswith()`, `between()` and `is_()` are methods for the SQL operators that
  Python has no symbol for.
- **Python's `and`, `or`, `not` and `is` cannot be changed.** A class can decide what `&` and `==`
  do, and has no say over `and` or `is`. So `and` between two conditions hands Python's own truth
  test to the first of them, and `is None` asks whether the column object is `None`, which it never
  is. Write `&`, `and_()`, or several conditions inside one `.where()`, and `.is_(None)`.
- **A write without `.where()` writes every row**, in SQLAlchemy exactly as in SQL.
- **Every statement compiles for a dialect.** `print(statement)` writes it for no database in
  particular, with placeholders such as `:program_1`, and `statement.compile(engine)` writes what
  the engine will send. `literal_binds` puts the values into the text, for a person to read.
- **Some statements belong to one database.** An upsert, an insert that updates the row when it is
  already there, is written differently by SQLite, PostgreSQL and MySQL, so it comes from the
  dialect's own `insert`, `sqlalchemy.dialects.sqlite.insert` for SQLite.
- **A statement runs once for every dictionary in a list.** That is `executemany`, and it is how a
  term's grades go to the database in one call.

### Where this shows up

The ORM's queries, from the **Declarative Models** notebook on, are these statements with classes in
place of tables: `select(Student).where(Student.program == "History")`. The **Joins and Aggregates**
notebook takes `join`, `func` and `group_by` further with classes. Alembic's migrations run
`update()` and `insert()` statements like these when they change data as well as tables. Pandas'
`read_sql` accepts a `select()`. The **SQL Syntax** notebook of the **sqlite3, Deep Dive** guide
covered the SQL that these statements become.

### What this notebook covers

- Reading the SQL a statement becomes, three ways
- `select` with `where`, `order_by` and `limit`
- Conditions: `&`, `|` and `~`, `in_`, `like`, `between` and `IS NULL`
- `func`, labels and `group_by`
- Joins that follow the foreign keys
- `insert`, with `returning()`
- `update` and `delete`, and how many rows they changed
- An upsert for an import that runs twice
- Which way to write a condition and a write
- Grades at the end of term, finished
- Six errors, from the `where` that lost half of itself to a `bindparam` named like a column

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
from sqlalchemy import Column, Integer, MetaData, String, Table, create_engine
from sqlalchemy import insert, select, update

courses = Table("courses", MetaData(), Column("id", Integer, primary_key=True),
                Column("code", String(10)), Column("credits", Integer))
engine = create_engine("sqlite://")
courses.metadata.create_all(engine)

with engine.begin() as conn:
    conn.execute(insert(courses), [{"code": "BIO-101", "credits": 4},
                                   {"code": "STA-200", "credits": 3}])
    add_one = (update(courses).where(courses.c.code == "STA-200")
               .values(credits=courses.c.credits + 1))
    print(add_one)
    print(conn.execute(add_one).rowcount, "row updated")
    print(conn.execute(select(courses.c.code, courses.c.credits).order_by(courses.c.code)).all())
```

```
UPDATE courses SET credits=(courses.credits + :credits_1) WHERE courses.code = :code_1
1 row updated
[('BIO-101', 4), ('STA-200', 4)]
```

An insert, an update and a select, each built from the table. The update's new value is an
expression on the column itself, `courses.c.credits + 1`, and printing the statement shows the SQL
with a placeholder for every value.


## Setup

Eight imports, the college built from its `MetaData`, and a helper that prints SQL.

- `sqlalchemy` is the library itself, and the cell prints its version
- `select`, `insert`, `update` and `delete`, `and_` and `or_`, `func` and `bindparam`, from
  `sqlalchemy`, build the statements, with `MetaData`, `Table`, `Column`, the types, the constraints
  and `ForeignKey` to describe the tables, and `create_engine` and `event` for the engine
- `sqlite_insert`, from `sqlalchemy.dialects.sqlite`, is SQLite's own `insert`, which can upsert
- `IntegrityError`, from `sqlalchemy.exc`, is the error a broken constraint raises
- `StaticPool`, from `sqlalchemy.pool`, is the pool the helper uses for a database in memory
- `date` is what the `Date` columns take and return
- `Path` names the files, and `shutil` removes the scratch folder at the start and at the end

From this notebook on, Setup builds the college the way the **Tables and Metadata** notebook ended:
`college` describes the five tables, `students`, `courses`, `terms`, `sections` and `enrollments`,
with named constraints, and `build_college` creates them in `scratch/college.db` and loads the lists
above it. `college_engine` is the engine helper, and `show_sql` is the helper from the
**Why SQLAlchemy** notebook, which prints the SQL a statement becomes for one database, and the
values that go with it.

Colab has SQLAlchemy installed, and this notebook runs version 2.0.54. Any 2.0 release runs it,
though an error may be worded a little differently. To match it exactly, run
`%pip install sqlalchemy==2.0.54` in a cell of its own, restart the session, and run this cell again.


In [1]:
import shutil
from datetime import date
from pathlib import Path

import sqlalchemy
from sqlalchemy import (CheckConstraint, Column, Date, ForeignKey, Integer, MetaData, String, Table, UniqueConstraint,
                        and_, bindparam, create_engine, delete, event, func, insert, or_, select, update)
from sqlalchemy.dialects.sqlite import insert as sqlite_insert
from sqlalchemy.exc import IntegrityError
from sqlalchemy.pool import StaticPool

SCRATCH = Path("scratch")
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir()
DATABASE = SCRATCH / "college.db"

NAMES = [
    "Ana Reyes", "Ben Okafor", "Chloe Martin", "Daniel Kim", "Elena Petrova", "Felix Wagner",
    "Grace Lin", "Hassan Ali", "Isabel Costa", "Jonas Berg", "Keiko Tanaka", "Liam Murphy",
    "Maya Patel", "Noah Andersen", "Olivia Brandt", "Pavel Novak", "Quinn Harper", "Rosa Delgado",
    "Sam Ito", "Tara Nilsen", "Umar Farouk", "Vera Kowalski", "Wes Carter", "Yara Haddad",
    "Aoife O'Brien",
]
PROGRAMS = ["Biology", "Computer Science", "Mathematics", "Psychology", "History"]
TERMS = [("Fall 2024", "2024-08-26"), ("Spring 2025", "2025-01-13"), ("Fall 2025", "2025-08-25"),
         ("Spring 2026", "2026-01-12")]
STUDENTS = [(name, f"{name[0]}{name.split()[-1]}@college.edu".lower().replace("'", ""),
             PROGRAMS[i % len(PROGRAMS)], TERMS[i % 3][1]) for i, name in enumerate(NAMES)]
COURSES = [
    ("BIO-101", "Introduction to Biology", "Biology", 4),
    ("CHE-110", "General Chemistry", "Chemistry", 4),
    ("MAT-120", "Calculus I", "Mathematics", 4),
    ("MAT-121", "Calculus II", "Mathematics", 4),
    ("CSC-101", "Programming I", "Computer Science", 3),
    ("CSC-201", "Data Structures", "Computer Science", 3),
    ("ENG-105", "Composition", "English", 3),
    ("HIS-110", "World History", "History", 3),
    ("PSY-101", "Introduction to Psychology", "Psychology", 3),
    ("STA-200", "Statistics", "Mathematics", 3),
]
GRADES = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "D", "F"]

# One section of every course in every term, so the section of course c in term t has id (t - 1) * 10 + c.
SECTIONS = [(course, term, 30) for term in range(1, len(TERMS) + 1) for course in range(1, len(COURSES) + 1)]

# Three courses a term for every student, from the term they started. Spring 2026 is under way.
ENROLLMENTS = []
for s in range(len(NAMES)):
    for term in range(s % 3 + 1, len(TERMS) + 1):
        for k in range(3):
            section = (term - 1) * len(COURSES) + (s + term + 3 * k) % len(COURSES) + 1
            if term < len(TERMS):
                ENROLLMENTS.append((s + 1, section, "completed", GRADES[(s * 7 + term * 5 + k * 3) % len(GRADES)]))
            else:
                ENROLLMENTS.append((s + 1, section, "enrolled", None))

def college_engine(path=None, echo=False):
    """An engine for the college's database, in a file or in memory, with foreign keys enforced."""
    if path is None:                                # in memory: one connection, and one database, for every thread
        engine = create_engine("sqlite://", poolclass=StaticPool, echo=echo,
                               connect_args={"check_same_thread": False, "autocommit": False})
    else:
        engine = create_engine(f"sqlite:///{path}", echo=echo, connect_args={"autocommit": False})

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(dbapi_connection, connection_record):
        dbapi_connection.autocommit = True          # the PRAGMA does nothing inside a transaction,
        dbapi_connection.execute("PRAGMA foreign_keys = ON")
        dbapi_connection.autocommit = False         # and with autocommit=False sqlite3 keeps one open

    return engine

NAMING = {
    "pk": "pk_%(table_name)s",
    "uq": "uq_%(table_name)s_%(column_0_N_name)s",
    "ck": "ck_%(table_name)s_%(constraint_name)s",
    "fk": "fk_%(table_name)s_%(column_0_name)s_%(referred_table_name)s",
    "ix": "ix_%(column_0_label)s",
}
college = MetaData(naming_convention=NAMING)

students = Table(
    "students", college,
    Column("id", Integer, primary_key=True),
    Column("name", String(100), nullable=False),
    Column("email", String(200), nullable=False, unique=True),
    Column("program", String(50), nullable=False),
    Column("started_on", Date, nullable=False),
)
courses = Table(
    "courses", college,
    Column("id", Integer, primary_key=True),
    Column("code", String(10), nullable=False, unique=True),
    Column("title", String(100), nullable=False),
    Column("department", String(50), nullable=False),
    Column("credits", Integer, nullable=False),
    CheckConstraint("credits BETWEEN 1 AND 6", name="credits_range"),
)
terms = Table(
    "terms", college,
    Column("id", Integer, primary_key=True),
    Column("name", String(20), nullable=False, unique=True),
    Column("starts_on", Date, nullable=False),
)
sections = Table(
    "sections", college,
    Column("id", Integer, primary_key=True),
    Column("course_id", ForeignKey("courses.id"), nullable=False),
    Column("term_id", ForeignKey("terms.id"), nullable=False),
    Column("capacity", Integer, nullable=False),
    UniqueConstraint("course_id", "term_id"),
    CheckConstraint("capacity > 0", name="capacity_positive"),
)
enrollments = Table(
    "enrollments", college,
    Column("student_id", ForeignKey("students.id"), primary_key=True),
    Column("section_id", ForeignKey("sections.id"), primary_key=True),
    Column("status", String(20), nullable=False, server_default="enrolled"),
    Column("grade", String(2)),
    CheckConstraint("status IN ('enrolled', 'completed', 'withdrawn')", name="status_known"),
)

def build_college(engine):
    """Create the college's tables from `college`, load the lists from Setup into them, and count their rows."""
    college.create_all(engine)
    rows = {
        students: [{"name": name, "email": email, "program": program, "started_on": date.fromisoformat(started)}
                   for name, email, program, started in STUDENTS],
        courses: [{"code": code, "title": title, "department": department, "credits": credits}
                  for code, title, department, credits in COURSES],
        terms: [{"name": name, "starts_on": date.fromisoformat(starts)} for name, starts in TERMS],
        sections: [{"course_id": course, "term_id": term, "capacity": capacity} for course, term, capacity in SECTIONS],
        enrollments: [{"student_id": student, "section_id": section, "status": status, "grade": grade}
                      for student, section, status, grade in ENROLLMENTS],
    }
    with engine.begin() as conn:
        for table in college.sorted_tables:
            conn.execute(insert(table), rows[table])
        return {table.name: conn.execute(select(func.count()).select_from(table)).scalar_one()
                for table in college.sorted_tables}


def show_sql(statement, dialect):
    """Print the SQL a statement becomes for one database, and the values that travel beside it."""
    compiled = statement.compile(dialect=dialect)
    for line in str(compiled).splitlines():
        print("   ", line.rstrip())
    print("    values:", compiled.params)

engine = college_engine(DATABASE)
print("sqlalchemy", sqlalchemy.__version__, "|", DATABASE, "|", build_college(engine))


sqlalchemy 2.0.54 | scratch/college.db | {'courses': 10, 'students': 25, 'terms': 4, 'sections': 40, 'enrollments': 228}


## Worked examples

### Reading the SQL a statement becomes

A statement can be printed three ways. `print()` writes it for no database in particular. `show_sql`
compiles it for the engine's dialect, which is what goes to SQLite, with the values beside it. And
`literal_binds` writes the values into the text:


In [2]:
query = select(students.c.name).where(students.c.program == "History").order_by(students.c.name).limit(2)

print(query)
print("--- for SQLite:")
show_sql(query, engine.dialect)
print("--- with the values written in, for reading only:")
print(query.compile(engine, compile_kwargs={"literal_binds": True}))


SELECT students.name 
FROM students 
WHERE students.program = :program_1 ORDER BY students.name
 LIMIT :param_1
--- for SQLite:
    SELECT students.name
    FROM students
    WHERE students.program = ? ORDER BY students.name
     LIMIT ? OFFSET ?
    values: {'program_1': 'History', 'param_1': 2, 'param_2': 0}
--- with the values written in, for reading only:
SELECT students.name 
FROM students 
WHERE students.program = 'History' ORDER BY students.name
 LIMIT 2 OFFSET 0


`print()` used named placeholders, `:program_1`, the way SQLAlchemy writes SQL when it does not know
the database. SQLite's driver takes `?`, and the values travel beside the SQL, as `show_sql` shows.
`literal_binds` pasted `'History'` into the text, which is useful for reading a long statement and
never for running one, since pasting values into SQL is the mistake the **Why SQLAlchemy** notebook
began with. SQLAlchemy writes `OFFSET` beside `LIMIT` for SQLite, which changes nothing.

### select, where, order_by and limit

Every call returns a new statement, so a query can be written in steps. `.desc()` sorts a column
from the largest down:


In [3]:
recent_history = (
    select(students.c.name, students.c.started_on)
    .where(students.c.program == "History")
    .where(students.c.started_on >= date(2025, 1, 1))
    .order_by(students.c.started_on.desc(), students.c.name)
)
show_sql(recent_history, engine.dialect)

with engine.connect() as conn:
    for row in conn.execute(recent_history):
        print(f"{row.name:<14}", row.started_on)


    SELECT students.name, students.started_on
    FROM students
    WHERE students.program = ? AND students.started_on >= ? ORDER BY students.started_on DESC, students.name
    values: {'program_1': 'History', 'started_on_1': datetime.date(2025, 1, 1)}
Olivia Brandt  2025-08-25
Elena Petrova  2025-01-13
Tara Nilsen    2025-01-13


Two `.where()` calls, joined by `AND`, and a date compared as a `date`, which the `Date` column turns
into the text SQLite stores. Three of the five History students started in 2025, the latest first,
and the two who started on the same day in name order.

### Conditions: &, |, ~, in_, like, between and IS NULL

Every condition here counts the students it matches. Each is also printed with its values written
in, which is the quickest way to check that a condition says what was meant:


In [4]:
CONDITIONS = [
    ("& is AND", (students.c.program == "Biology") & (students.c.started_on >= date(2025, 1, 1))),
    ("| is OR", (students.c.program == "Biology") | (students.c.program == "History")),
    ("in_ is IN", students.c.program.in_(["Biology", "History"])),
    ("~ is NOT", ~students.c.program.in_(["Biology", "History"])),
    ("startswith is LIKE", students.c.name.startswith("A")),
    ("between", students.c.started_on.between(date(2025, 1, 1), date(2025, 6, 30))),
]

with engine.connect() as conn:
    for label, condition in CONDITIONS:
        count = conn.execute(select(func.count()).select_from(students).where(condition)).scalar_one()
        print(f"{label:<19} {count:>2} | {condition.compile(engine, compile_kwargs={'literal_binds': True})}")


& is AND             3 | students.program = 'Biology' AND students.started_on >= '2025-01-01'
| is OR             10 | students.program = 'Biology' OR students.program = 'History'
in_ is IN           10 | students.program IN ('Biology', 'History')
~ is NOT            15 | (students.program NOT IN ('Biology', 'History'))
startswith is LIKE   2 | students.name LIKE 'A' || '%'
between              8 | students.started_on BETWEEN '2025-01-01' AND '2025-06-30'


`&`, `|` and `~` are Python's operators for bits, which SQLAlchemy turns into `AND`, `OR` and `NOT`,
and they bind more tightly than `==`, so every comparison next to one needs its own parentheses.
`and_()`, `or_()` and `not_()` do the same as functions, and `.where(a, b)` is the same as
`.where(a & b)`. `in_` takes a list, `startswith` writes a `LIKE` with the `%` added, and `between`
includes both ends. A missing value is `NULL`, and SQL finds it with `IS NULL`, which `.is_(None)`
writes:


In [5]:
no_grade = select(func.count()).select_from(enrollments).where(enrollments.c.grade.is_(None))
show_sql(no_grade, engine.dialect)
with engine.connect() as conn:
    print(conn.execute(no_grade).scalar_one(), "enrollments with no grade yet")


    SELECT count(*) AS count_1
    FROM enrollments
    WHERE enrollments.grade IS NULL
    values: {}
75 enrollments with no grade yet


Seventy-five: three courses for each of the 25 students in Spring 2026, which has no grades yet.
`enrollments.c.grade == None` writes the same `IS NULL`, and `is None` in its place is one of the
Common errors.

### func, labels and group_by

`func.count()` is SQL's `COUNT(*)`, and any other name after `func.` becomes the SQL function of that
name. `.label()` names a column of the result, and `group_by` gives one row for every group:


In [6]:
fall_grades = (
    select(enrollments.c.grade, func.count().label("students"))
    .where(enrollments.c.section_id.between(21, 30))              # the sections of Fall 2025
    .group_by(enrollments.c.grade)
    .order_by(func.count().desc(), enrollments.c.grade)
)
with engine.connect() as conn:
    first, last = conn.execute(select(func.min(students.c.started_on), func.max(students.c.started_on))).one()
    print("first days run from", repr(first), "to", repr(last))
    for row in conn.execute(fall_grades):
        print(f"{row.grade:<3} {row.students}")


first days run from datetime.date(2024, 8, 26) to datetime.date(2025, 8, 25)
B+  9
C+  9
F   9
C   8
D   8
A-  7
B   7
A   6
B-  6
C-  6


`func.min` and `func.max` of a `Date` column came back as `date` objects, because SQLAlchemy gives
`min` and `max` the type of the column they are applied to. The label became the name of the count,
`row.students`, and the grades of Fall 2025 are grouped and ordered by how many students got each,
the most common first. The **Joins and Aggregates** notebook does more with groups.

### Joins that follow the foreign keys

`.join(sections)` joins the table to the one before it, and when a foreign key links the two, it
writes the `ON` clause from the key. Chloe Martin's transcript, the query the **Reading Results**
notebook wrote as text, built from the tables:


In [7]:
transcript = (
    select(terms.c.name.label("term"), courses.c.code, enrollments.c.grade)
    .select_from(enrollments)
    .join(sections)                     # ON sections.id = enrollments.section_id, from the foreign key
    .join(courses)
    .join(terms)
    .where(enrollments.c.student_id == 3)
    .order_by(terms.c.starts_on, courses.c.code)
)
show_sql(transcript, engine.dialect)

with engine.connect() as conn:
    for row in conn.execute(transcript):
        print(row.term, row.code, row.grade)


    SELECT terms.name AS term, courses.code, enrollments.grade
    FROM enrollments JOIN sections ON sections.id = enrollments.section_id JOIN courses ON courses.id = sections.course_id JOIN terms ON terms.id = sections.term_id
    WHERE enrollments.student_id = ? ORDER BY terms.starts_on, courses.code
    values: {'student_id_1': 3}
Fall 2025 CHE-110 C+
Fall 2025 CSC-201 F
Fall 2025 PSY-101 B+
Spring 2026 ENG-105 None
Spring 2026 MAT-120 None
Spring 2026 STA-200 None


`select_from(enrollments)` names the table the joins start from, and every `.join()` found the
foreign key to follow: `sections` from `enrollments`, and `courses` and `terms` from `sections`. A
join that no foreign key explains takes its condition as a second argument,
`.join(courses, courses.c.id == sections.c.course_id)`.

### insert, with returning()

A new student is admitted for Spring 2026. `.values()` gives the row, and `.returning()` asks for the
id the database gave it, which the enrollments then use. A list of dictionaries inserts several rows
in one call:


In [8]:
admit = insert(students).values(name="Zoe Nakamura", email="znakamura@college.edu",
                                program="Computer Science", started_on=date(2026, 1, 12)).returning(students.c.id)
show_sql(admit, engine.dialect)

with engine.begin() as conn:
    zoe = conn.execute(admit).scalar_one()
    added = conn.execute(insert(enrollments), [{"student_id": zoe, "section_id": section} for section in (35, 36, 40)])
print("Zoe Nakamura is student", zoe, "| enrollments added:", added.rowcount)


    INSERT INTO students (name, email, program, started_on) VALUES (?, ?, ?, ?) RETURNING id
    values: {'name': 'Zoe Nakamura', 'email': 'znakamura@college.edu', 'program': 'Computer Science', 'started_on': datetime.date(2026, 1, 12)}
Zoe Nakamura is student 26 | enrollments added: 3


`RETURNING id` came back with 26, the next id after the 25 students, and the three enrollments went
in with one `INSERT` run three times. None of them named a status, so the table's
`DEFAULT 'enrolled'` filled it in. `rowcount` is the number of rows a write changed, here the three
together.

### update and delete, and how many rows they changed

The Computer Science sections of Spring 2026 need five more seats each. The new value is an
expression on the column, and the sections to change come from a subquery, a `select` used inside
the condition:


In [9]:
more_seats = (
    update(sections)
    .where(sections.c.term_id == 4)
    .where(sections.c.course_id.in_(select(courses.c.id).where(courses.c.department == "Computer Science")))
    .values(capacity=sections.c.capacity + 5)
)
show_sql(more_seats, engine.dialect)

with engine.begin() as conn:
    print("sections changed:", conn.execute(more_seats).rowcount)
    print(conn.execute(select(sections.c.id, sections.c.capacity).where(sections.c.id.in_([35, 36, 37]))).all())


    UPDATE sections SET capacity=(sections.capacity + ?) WHERE sections.term_id = ? AND sections.course_id IN (SELECT courses.id
    FROM courses
    WHERE courses.department = ?)
    values: {'capacity_1': 5, 'term_id_1': 4, 'department_1': 'Computer Science'}
sections changed: 2
[(35, 35), (36, 35), (37, 30)]


Two sections changed, 35 and 36, from 30 seats to 35, and section 37, for English, kept its 30.
`capacity + ?` is computed by the database, row by row, which is safe even when two programs change
the same row at once. Reading the capacity, adding five in Python, and writing it back is not. Zoe
Nakamura withdraws from Statistics, and `delete` removes the one enrollment:


In [10]:
withdraw = delete(enrollments).where(enrollments.c.student_id == zoe, enrollments.c.section_id == 40)
show_sql(withdraw, engine.dialect)

with engine.begin() as conn:
    print("rows deleted:", conn.execute(withdraw).rowcount)


    DELETE FROM enrollments WHERE enrollments.student_id = ? AND enrollments.section_id = ?
    values: {'student_id_1': 26, 'section_id_1': 40}
rows deleted: 1


### An upsert for an import that runs twice

The mathematics department sends Chloe Martin's two finished grades early. A plain `insert` of an
enrollment that already exists breaks the primary key, and so does running the import a second time.
SQLite's `INSERT ... ON CONFLICT DO UPDATE` writes the row when it is new and updates it when it is
not, and SQLite's own `insert` builds one, with `excluded` standing for the row that was refused:


In [11]:
SHEET = [
    {"student_id": 3, "section_id": 33, "status": "completed", "grade": "A-"},     # Calculus I
    {"student_id": 3, "section_id": 40, "status": "completed", "grade": "B"},      # Statistics
]

try:
    with engine.begin() as conn:
        conn.execute(insert(enrollments), SHEET)
except IntegrityError as error:
    print("a plain insert:", error.orig)

upsert = sqlite_insert(enrollments)
upsert = upsert.on_conflict_do_update(
    index_elements=[enrollments.c.student_id, enrollments.c.section_id],
    set_={"status": upsert.excluded.status, "grade": upsert.excluded.grade},
)
print(upsert.compile(engine))

with engine.begin() as conn:
    print("rows written, the first run: ", conn.execute(upsert, SHEET).rowcount)
    print("rows written, the second run:", conn.execute(upsert, SHEET).rowcount)
    both = select(enrollments).where(enrollments.c.student_id == 3, enrollments.c.section_id.in_([33, 40]))
    print(conn.execute(both).all())


a plain insert: UNIQUE constraint failed: enrollments.student_id, enrollments.section_id
INSERT INTO enrollments (student_id, section_id, status, grade) VALUES (?, ?, ?, ?) ON CONFLICT (student_id, section_id) DO UPDATE SET status = excluded.status, grade = excluded.grade
rows written, the first run:  2
rows written, the second run: 2
[(3, 33, 'completed', 'A-'), (3, 40, 'completed', 'B')]


The plain insert was refused on its first row. The upsert wrote both rows twice, the second time
changing nothing, and a job that writes the same grades again leaves them as they were. The
statement is SQLite's: `sqlalchemy.dialects.postgresql.insert` writes PostgreSQL's version, with the
same method, and MySQL's `insert` has `on_duplicate_key_update` instead.

### Which way to write a condition, and a write

| Use | When | Why |
|---|---|---|
| `.where(a, b)`, or `.where()` called twice | every condition must hold | the plainest way to write `AND` |
| `a & b`, `a \| b`, `~a`, each comparison in parentheses | a condition built from parts, such as `(a & b) \| c` | reads like the SQL it becomes |
| `and_(...)`, `or_(...)`, `not_(...)` | a list of conditions built in a loop, as `or_(*conditions)` | takes any number of conditions |
| `.is_(None)` and `.is_not(None)` | a column that may be `NULL` | `is None` is Python's, and never reaches SQL |
| `insert()` with a list of dictionaries | many rows | one statement, run once for every row |
| `.returning(...)` | a write whose result the program needs, such as a new id | no second query |
| the dialect's `insert` with `on_conflict_do_update` | a load that may run twice | new rows are written, and existing rows updated |
| `bindparam("name")` | one statement run with many sets of values | the values arrive when the statement runs |

The default is several conditions inside one `.where()`, `&` and `|` when the logic is more than a
list, and `.is_(None)` for every `NULL`.

### Grades at the end of term, finished

The pieces of this notebook in one function. `record_grades` takes a term's grade sheet, with every
course named by its code. `SECTION_OF` is a join that turns codes into the term's sections, and
`GRADE` is an `update` whose student, section and grade are `bindparam`s, run once for every line of
the sheet.
`STILL_OPEN` counts what the term has left without a grade. The names of the bind parameters differ
from the columns' names, which Common errors shows is required:


In [12]:
SECTION_OF = (                                   # course code -> section, for one term
    select(courses.c.code, sections.c.id)
    .select_from(sections)
    .join(courses)
    .join(terms)
    .where(terms.c.name == bindparam("term"))
)
GRADE = (
    update(enrollments)
    .where(enrollments.c.student_id == bindparam("student"), enrollments.c.section_id == bindparam("section"))
    .values(grade=bindparam("letter"), status="completed")
)
STILL_OPEN = (
    select(func.count())
    .select_from(enrollments.join(sections).join(terms))
    .where(terms.c.name == bindparam("term"), enrollments.c.grade.is_(None))
)


def record_grades(engine, term, sheet):
    """Record a term's grade sheet of (student, course code, grade), and count what was graded and what is still open."""
    with engine.begin() as conn:
        section_of = dict(conn.execute(SECTION_OF, {"term": term}).all())
        rows = [{"student": student, "section": section_of[code], "letter": letter} for student, code, letter in sheet]
        graded = conn.execute(GRADE, rows).rowcount
        return graded, conn.execute(STILL_OPEN, {"term": term}).scalar_one()



SPRING_SHEET = [
    (1, "BIO-101", "A"), (1, "CSC-101", "B+"), (1, "HIS-110", "A-"),       # Ana Reyes
    (2, "CHE-110", "B"), (2, "CSC-201", "C+"), (2, "PSY-101", "A"),        # Ben Okafor
]
graded, still_open = record_grades(engine, "Spring 2026", SPRING_SHEET)
print("graded:", graded, "| still without a grade:", still_open)

ANA_SPRING = (
    select(courses.c.code, enrollments.c.status, enrollments.c.grade)
    .select_from(enrollments)
    .join(sections)
    .join(courses)
    .where(enrollments.c.student_id == 1, sections.c.term_id == 4)
    .order_by(courses.c.code)
)
with engine.connect() as conn:
    print("Ana Reyes:", conn.execute(ANA_SPRING).all())


graded: 6 | still without a grade: 69
Ana Reyes: [('BIO-101', 'completed', 'A'), ('CSC-101', 'completed', 'B+'), ('HIS-110', 'completed', 'A-')]


Six lines of the sheet, six enrollments graded and completed, and 69 of Spring 2026's enrollments
still waiting: the 75 counted above, plus the two of Zoe Nakamura's that remain, less Chloe Martin's
two from the upsert and the six graded here. `GRADE` ran once with all six dictionaries, and
`rowcount` added up the rows it changed.

### Where each part came from

| In `record_grades` | What it relies on | The section that showed it |
|---|---|---|
| `SECTION_OF`, a `select` with `.join()` | joins that follow the foreign keys | Joins that follow the foreign keys |
| `.where(a, b)` in `GRADE` | several conditions joined by `AND` | Conditions: &, \|, ~, in_, like, between and IS NULL |
| `update(...).values(...)` and `rowcount` | a write, and how many rows it changed | update and delete, and how many rows they changed |
| `bindparam(...)`, run with a list of dictionaries | one statement, many sets of values | Grades at the end of term, finished |
| `.is_(None)` in `STILL_OPEN` | `IS NULL`, for the grades not given yet | Conditions: &, \|, ~, in_, like, between and IS NULL |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/06-sql-expressions-solutions.ipynb).

**1.** Select the Mathematics courses worth 4 credits, by code, with two conditions inside one
`.where()`. Print the SQL with `show_sql`, then the rows.


In [13]:
# your code here


**2.** List the students whose name holds an apostrophe, with `contains`.


In [14]:
# your code here


**3.** Count the enrollments of Fall 2025, sections 21 to 30, that ended with a grade of D or F, with
`between` and `in_`.


In [15]:
# your code here


**4.** Add a course, `ART-100 Drawing`, in the Art department and worth 3 credits, and use
`returning()` to give it a Spring 2026 section of 20 seats. Print both new ids.


In [16]:
# your code here


**5.** Remove that section and its course with two `delete` statements in one transaction, in an
order the foreign keys accept, and print both `rowcount`s.


In [17]:
# your code here


**6.** Print the SQL of `GRADE`, the `update` inside `record_grades`, for SQLite and for PostgreSQL,
and say what the placeholders look like in each.


In [18]:
# your code here


## Common errors

### No error, and half the filter gone: two conditions joined with and


In [19]:
started_with_fall_2025 = select(students.c.name).where(
    students.c.program == "Biology" and students.c.started_on == date(2025, 8, 25)
)
show_sql(started_with_fall_2025, engine.dialect)
with engine.connect() as conn:
    print(conn.execute(started_with_fall_2025).scalars().all())


    SELECT students.name
    FROM students
    WHERE students.program = ?
    values: {'program_1': 'Biology'}
['Ana Reyes', 'Felix Wagner', 'Keiko Tanaka', 'Pavel Novak', 'Umar Farouk']


Two conditions were written, and the SQL has one: every Biology student came back, five of them,
where two started on 25 August 2025. Python worked out `a and b` before SQLAlchemy saw anything.
`and` asks whether `a` is true, and SQLAlchemy gives an `==` a truth value, `True` only when both
sides are the same object, which is what lets Python find a column in a list. The two sides here are
a column and a value, so `a` was false, and `and` returned it and threw `b` away. Write the
conditions as two arguments to `.where()`, or with `&`:


In [20]:
started_with_fall_2025 = select(students.c.name).where(
    students.c.program == "Biology", students.c.started_on == date(2025, 8, 25)
)
show_sql(started_with_fall_2025, engine.dialect)
with engine.connect() as conn:
    print(conn.execute(started_with_fall_2025).scalars().all())


    SELECT students.name
    FROM students
    WHERE students.program = ? AND students.started_on = ?
    values: {'program_1': 'Biology', 'started_on_1': datetime.date(2025, 8, 25)}
['Felix Wagner', 'Umar Farouk']


### TypeError: Boolean value of this clause is not defined


In [21]:
select(students.c.name).where(students.c.started_on > date(2025, 1, 1) and students.c.program == "Biology")


TypeError: Boolean value of this clause is not defined

The same `and`, with a `>` in front. An `==` has a truth value, and a `>` has none, so this time
`and` raised instead of quietly dropping half of the condition. The error is the better outcome of
the two, and the fix is the same, `&` with parentheses or two arguments:


In [22]:
with engine.connect() as conn:
    print(conn.execute(select(students.c.name).where(
        (students.c.started_on > date(2025, 1, 1)) & (students.c.program == "Biology")
    ).order_by(students.c.name)).scalars().all())


['Felix Wagner', 'Keiko Tanaka', 'Umar Farouk']


### No error, and no rows: is None in a where clause


In [23]:
ungraded = select(func.count()).select_from(enrollments).where(enrollments.c.grade is None)
show_sql(ungraded, engine.dialect)
with engine.connect() as conn:
    print(conn.execute(ungraded).scalar_one(), "enrollments with no grade")


    SELECT count(*) AS count_1
    FROM enrollments
    WHERE 0 = 1
    values: {}
0 enrollments with no grade


`enrollments.c.grade is None` is Python asking whether the column object is `None`, which it is not,
so `.where()` received `False`, and SQLAlchemy wrote a condition that is always false, `0 = 1`. The
count was zero, and nothing raised. SQL's `IS NULL` comes from `.is_(None)`, or from `== None`:


In [24]:
ungraded = select(func.count()).select_from(enrollments).where(enrollments.c.grade.is_(None))
with engine.connect() as conn:
    print(conn.execute(ungraded).scalar_one(), "enrollments with no grade")


69 enrollments with no grade


### sqlalchemy.exc.ArgumentError: IN expression list, SELECT construct, or bound parameter object expected, got 'History'.


In [25]:
select(students.c.name).where(students.c.program.in_("History"))


ArgumentError: IN expression list, SELECT construct, or bound parameter object expected, got 'History'.

`in_` takes a list of values, or a `select` whose rows are the values, and a string is neither.
SQLAlchemy refuses it rather than treat the string as a list of seven letters. Put one value in a
list, or compare with `==`:


In [26]:
with engine.connect() as conn:
    print(conn.execute(select(func.count()).select_from(students).where(students.c.program.in_(["History"]))).scalar_one())


5


### No error, and every row changed: an update with no where


In [27]:
with engine.connect() as conn:                   # a trial run: nothing in this block is committed
    changed = conn.execute(update(students).values(program="Mathematics"))       # meant for Zoe Nakamura
    print("rows changed:", changed.rowcount)


rows changed: 26


The update was meant for one student, and it changed all 26, because an `update` with no `.where()`
changes every row, in SQLAlchemy exactly as in SQL. This block never committed, so its rollback put
everything back. `rowcount` is the check worth writing into any update meant for one row:


In [28]:
with engine.begin() as conn:
    changed = conn.execute(update(students).where(students.c.id == zoe).values(program="Mathematics"))
    print("rows changed:", changed.rowcount)


rows changed: 1


### sqlalchemy.exc.CompileError: bindparam() name 'student_id' is reserved for automatic usage in the VALUES or SET clause of this insert/update statement.   Please use a name other than column name when using bindparam() with insert() or update() (for example, 'b_student_id').


In [29]:
grade_by_column_names = (
    update(enrollments)
    .where(enrollments.c.student_id == bindparam("student_id"), enrollments.c.section_id == bindparam("section_id"))
    .values(grade=bindparam("grade"))
)
with engine.begin() as conn:
    conn.execute(grade_by_column_names, [{"student_id": 3, "section_id": 37, "grade": "B+"}])


CompileError: bindparam() name 'student_id' is reserved for automatic usage in the VALUES or SET clause of this insert/update statement.   Please use a name other than column name when using bindparam() with insert() or update() (for example, 'b_student_id').

In an `insert` or an `update`, SQLAlchemy uses every column's own name for the value that goes into
that column, so a `bindparam` may not borrow it. The statement compiled, and failed when it ran with
a list of dictionaries, whose keys it could not tell apart from the columns. The message suggests a
prefix, and any other names will do, which is why `record_grades` says `student`, `section` and
`letter`:


In [30]:
with engine.begin() as conn:
    print(conn.execute(GRADE, [{"student": 3, "section": 37, "letter": "B+"}]).rowcount, "row graded")


1 row graded


Last, the engine lets go of the file, and this cell removes the scratch folder, with the database in
it:


In [31]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- `select`, `insert`, `update` and `delete` build statements from tables, and every method returns a
  new statement.
- Conditions combine with several arguments to `.where()`, with `&`, `|` and `~` around
  parenthesized comparisons, or with `and_()`, `or_()` and `not_()`, never with Python's `and`,
  `or`, `not` or `is`.
- `.join()` follows a foreign key, `func` calls SQL functions, and `.label()` names a result's
  columns.
- `.returning()` gives a write's rows back, `rowcount` says how many it changed, and a write with no
  `.where()` changes every row.
- A dialect's own `insert` writes an upsert, and `bindparam` lets one statement run with a list of
  values.


## What is next

The **Declarative Models** notebook writes the college as Python classes, with `DeclarativeBase`,
`Mapped` and `mapped_column`, so that rows come back as objects, and shows the annotation it cannot
map.


---

&#8592; **Previous:** [Tables and Metadata](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/05-tables-and-metadata.ipynb)  &nbsp;·&nbsp;  [SQLAlchemy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)
